# DSA 210 – What Makes a Song Popular on Spotify?
**Ada Derviş Sarabil – 34362** | Spring 2026

Bu notebook 5 aşamadan oluşuyor:
1. Veri yükleme ve temizleme
2. VADER ile lyrics sentiment analizi
3. Yeni değişkenler türetme
4. EDA (Keşifsel Veri Analizi)
5. Hipotez testleri

---
## 1. Setup ve Import

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
import os

import nltk
nltk.download('vader_lexicon', quiet=True)
from nltk.sentiment.vader import SentimentIntensityAnalyzer

warnings.filterwarnings('ignore')
os.makedirs('figures', exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'sans-serif',
    'axes.spines.top': False,
    'axes.spines.right': False
})

SPOTIFY_GREEN = '#1DB954'
print('Hazır!')

---
## 2. Veri Yükleme ve Temizleme

In [ ]:
df = pd.read_csv('data/spotify_songs.csv')

# Kolon isimlerini düzelt
df.rename(columns={
    'track_popularity': 'popularity',
    'track_artist': 'artist',
    'playlist_genre': 'genre'
}, inplace=True)

# Aynı şarkı birden fazla playlist'te olabilir, track_id'ye göre tekilleştir
df.drop_duplicates(subset='track_id', keep='first', inplace=True)

print(f'Toplam şarkı: {len(df)}')
df.head(3)

In [ ]:
# VADER İngilizce için eğitildiğinden sadece İngilizce şarkıları al
df_en = df[df['language'] == 'en'].copy()

# Enstrümantal şarkıları işaretle (lyrics yok veya çok kısa)
df_en['is_instrumental'] = df_en['lyrics'].isnull() | (df_en['lyrics'].str.len() < 20)

print(f'İngilizce şarkı sayısı: {len(df_en)}')
print(f'Enstrümantal (lyrics yok): {df_en["is_instrumental"].sum()}')
print(f'Eksik değerler:\n{df_en[["popularity", "valence", "lyrics"]].isnull().sum()}')

---
## 3. VADER Sentiment Analizi

VADER, her şarkının lyrics'ini okuyup -1 ile +1 arasında bir skor üretiyor:
- **+1'e yakın** → pozitif, mutlu sözler
- **-1'e yakın** → negatif, karanlık sözler
- **0 civarı** → nötr

In [ ]:
sid = SentimentIntensityAnalyzer()

def get_sentiment(text):
    if pd.isnull(text) or len(str(text)) < 20:
        return np.nan
    return sid.polarity_scores(str(text))['compound']

print('Sentiment hesaplanıyor... (~30 saniye sürebilir)')
df_en['compound_sentiment'] = df_en['lyrics'].apply(get_sentiment)

print(f"Sentiment hesaplanan şarkı: {df_en['compound_sentiment'].notna().sum()}")
df_en['compound_sentiment'].describe()

---
## 4. Yeni Değişkenler Türetme

In [ ]:
# Sadece sentiment ve popularity'si olan şarkılarla çalış
df_sent = df_en[df_en['compound_sentiment'].notna() & df_en['popularity'].notna()].copy()

# Sentiment'i [0,1] aralığına normalize et (valence ile karşılaştırabilmek için)
df_sent['sentiment_norm'] = (df_sent['compound_sentiment'] + 1) / 2

# Valence-sentiment gap: müzik ile sözlerin duygusal farkı
df_sent['valence_sentiment_gap'] = (df_sent['valence'] - df_sent['sentiment_norm']).abs()

# Sentiment kategorisi
def sentiment_category(score):
    if score >= 0.05:
        return 'Positive'
    elif score <= -0.05:
        return 'Negative'
    else:
        return 'Neutral'

df_sent['sentiment_cat'] = df_sent['compound_sentiment'].apply(sentiment_category)

# Gap için median'a göre yüksek/düşük grupları
gap_median = df_sent['valence_sentiment_gap'].median()
df_sent['gap_group'] = df_sent['valence_sentiment_gap'].apply(
    lambda x: 'High gap' if x >= gap_median else 'Low gap'
)

print(f'Analiz dataseti: {len(df_sent)} şarkı')
print(f'Sentiment dağılımı:\n{df_sent["sentiment_cat"].value_counts()}')
print(f'Gap medyanı: {gap_median:.3f}')

---
## 5. EDA – Keşifsel Veri Analizi

### 5.1 Popularity Dağılımı

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df_sent['popularity'], bins=40, color=SPOTIFY_GREEN, edgecolor='white', linewidth=0.4)
axes[0].set_xlabel('Popularity score')
axes[0].set_ylabel('Şarkı sayısı')
axes[0].set_title('Popularity dağılımı')

genre_order = df_sent.groupby('genre')['popularity'].median().sort_values(ascending=False).index
data = [df_sent[df_sent['genre'] == g]['popularity'].values for g in genre_order]
bp = axes[1].boxplot(data, patch_artist=True, labels=genre_order,
                     medianprops=dict(color='black', linewidth=2),
                     flierprops=dict(marker='.', markersize=2, alpha=0.3))
for patch in bp['boxes']:
    patch.set_facecolor(SPOTIFY_GREEN)
    patch.set_alpha(0.6)
axes[1].set_title("Genre'e göre popularity")
axes[1].set_xlabel('Genre')
axes[1].set_ylabel('Popularity')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('figures/popularity_distribution.png', bbox_inches='tight')
plt.show()

### 5.2 Lyrics Sentiment Dağılımı

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df_sent['compound_sentiment'], bins=50, color='#6C63FF', edgecolor='white', linewidth=0.4)
axes[0].axvline(0, color='red', linestyle='--', linewidth=1, label='Nötr (0)')
axes[0].set_xlabel('VADER compound score')
axes[0].set_ylabel('Şarkı sayısı')
axes[0].set_title('Lyrics sentiment dağılımı')
axes[0].legend()

genre_sent = df_sent.groupby('genre')['compound_sentiment'].mean().sort_values()
colors = [SPOTIFY_GREEN if v >= 0 else '#E74C3C' for v in genre_sent.values]
axes[1].barh(genre_sent.index, genre_sent.values, color=colors)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_xlabel('Ortalama VADER skoru')
axes[1].set_title("Genre'e göre ortalama lyrics sentiment")

plt.tight_layout()
plt.savefig('figures/sentiment_distribution.png', bbox_inches='tight')
plt.show()

### 5.3 Valence vs. Sentiment – Ses ve Sözler Ne Kadar Uyuşuyor?

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

sc = ax.scatter(
    df_sent['valence'],
    df_sent['sentiment_norm'],
    c=df_sent['popularity'],
    cmap='RdYlGn',
    alpha=0.3,
    s=10,
    linewidths=0
)

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5, label='Tam uyum')
ax.text(0.78, 0.08, 'Mutlu ses\nKaranlık sözler', fontsize=8, color='gray', ha='center')
ax.text(0.18, 0.92, 'Hüzünlü ses\nPozitif sözler', fontsize=8, color='gray', ha='center')

plt.colorbar(sc, ax=ax, label='Popularity')
corr = df_sent['valence'].corr(df_sent['sentiment_norm'])
ax.set_xlabel(f'Valence (ses tabanlı, 0–1)\nKorelasyon: r = {corr:.3f}')
ax.set_ylabel('Lyrics sentiment (normalize, 0–1)')
ax.set_title('Valence vs. Lyrics Sentiment\n(renk = popularity)')
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('figures/valence_vs_sentiment.png', bbox_inches='tight')
plt.show()
print(f'Valence ile lyrics sentiment korelasyonu: r = {corr:.3f}')

### 5.4 Sentiment Kategorisine Göre Popularity

In [ ]:
cat_order = ['Negative', 'Neutral', 'Positive']
palette = {'Negative': '#E74C3C', 'Neutral': '#95A5A6', 'Positive': SPOTIFY_GREEN}

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

data_by_cat = [df_sent[df_sent['sentiment_cat'] == c]['popularity'].values for c in cat_order]
bp = axes[0].boxplot(data_by_cat, patch_artist=True, labels=cat_order,
                     medianprops=dict(color='black', linewidth=2),
                     flierprops=dict(marker='.', markersize=2, alpha=0.3))
for patch, cat in zip(bp['boxes'], cat_order):
    patch.set_facecolor(palette[cat])
    patch.set_alpha(0.7)
axes[0].set_ylabel('Popularity')
axes[0].set_title('Sentiment kategorisine göre popularity')

means = df_sent.groupby('sentiment_cat')['popularity'].mean().reindex(cat_order)
bars = axes[1].bar(cat_order, means.values, color=[palette[c] for c in cat_order], alpha=0.8)
for bar, val in zip(bars, means.values):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                 f'{val:.1f}', ha='center', va='bottom', fontsize=10)
axes[1].set_ylabel('Ortalama popularity')
axes[1].set_title('Ortalama popularity – sentiment kategorisi')
axes[1].set_ylim(0, means.max() * 1.15)

plt.tight_layout()
plt.savefig('figures/popularity_by_sentiment_category.png', bbox_inches='tight')
plt.show()

### 5.5 Valence–Sentiment Gap vs. Popularity

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(df_sent['valence_sentiment_gap'], df_sent['popularity'],
                alpha=0.2, s=8, color='#6C63FF', linewidths=0)
z = np.polyfit(df_sent['valence_sentiment_gap'], df_sent['popularity'], 1)
p_line = np.poly1d(z)
x_line = np.linspace(0, 1, 100)
axes[0].plot(x_line, p_line(x_line), 'r-', linewidth=1.5, label='Trend')
axes[0].set_xlabel('Valence–sentiment gap')
axes[0].set_ylabel('Popularity')
axes[0].set_title('Duygusal uyumsuzluk vs. popularity')
axes[0].legend()

gap_data = [df_sent[df_sent['gap_group'] == g]['popularity'].values for g in ['Low gap', 'High gap']]
bp2 = axes[1].boxplot(gap_data, patch_artist=True,
                      labels=['Low gap\n(ses≈sözler)', 'High gap\n(ses≠sözler)'],
                      medianprops=dict(color='black', linewidth=2),
                      flierprops=dict(marker='.', markersize=2, alpha=0.3))
bp2['boxes'][0].set_facecolor('#95A5A6'); bp2['boxes'][0].set_alpha(0.7)
bp2['boxes'][1].set_facecolor('#E74C3C'); bp2['boxes'][1].set_alpha(0.7)
axes[1].set_ylabel('Popularity')
axes[1].set_title('Düşük vs. yüksek duygusal uyumsuzluk')

plt.tight_layout()
plt.savefig('figures/valence_sentiment_gap.png', bbox_inches='tight')
plt.show()

### 5.6 Korelasyon Heatmap'i

In [ ]:
features = [
    'popularity', 'compound_sentiment', 'valence', 'valence_sentiment_gap',
    'danceability', 'energy', 'loudness', 'tempo',
    'acousticness', 'instrumentalness', 'speechiness', 'liveness'
]

corr_matrix = df_sent[features].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, vmin=-1, vmax=1,
            linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Korelasyon matrisi – audio features, sentiment ve popularity', pad=12)
plt.tight_layout()
plt.savefig('figures/correlation_heatmap.png', bbox_inches='tight')
plt.show()

---
## 6. Hipotez Testleri

Tüm testlerde α = 0.05 kullanılmaktadır.

### H1: Pozitif sözlü şarkılar daha mı popüler?

In [ ]:
pos = df_sent[df_sent['sentiment_cat'] == 'Positive']['popularity']
neg = df_sent[df_sent['sentiment_cat'] == 'Negative']['popularity']

stat_h1, p_h1 = stats.mannwhitneyu(pos, neg, alternative='greater')

print('=== H1: Pozitif lyrics → daha yüksek popularity? ===')
print(f'Pozitif şarkılar: n={len(pos)}, medyan popularity = {pos.median():.1f}')
print(f'Negatif şarkılar: n={len(neg)}, medyan popularity = {neg.median():.1f}')
print(f'Mann-Whitney U = {stat_h1:.0f},  p = {p_h1:.4f}')
if p_h1 < 0.05:
    print('Sonuç: H0 REDDEDİLDİ ✓ — pozitif sözlü şarkılar anlamlı şekilde daha popüler.')
else:
    print('Sonuç: H0 REDDEDİLEMEDİ — iki grup arasında anlamlı fark yok.')

### H2: Lyrics sentiment mi, yoksa valence mi popularity'yi daha iyi açıklıyor?

In [ ]:
r_sentiment, p_rs = stats.spearmanr(df_sent['compound_sentiment'], df_sent['popularity'])
r_valence,   p_rv = stats.spearmanr(df_sent['valence'],            df_sent['popularity'])
r_sv,        _    = stats.spearmanr(df_sent['compound_sentiment'], df_sent['valence'])

def steigers_test(r12, r13, r23, n):
    """İki bağımlı korelasyonun anlamlı olarak farklı olup olmadığını test eder."""
    R = 1 - r12**2 - r13**2 - r23**2 + 2*r12*r13*r23
    k = (r12**2 + r13**2) / 2
    denom_sq = 2*R**2 / (n-1) + (1-k)**2 * (1+r23)**2 / (2*(n-1)*(n-3))
    if denom_sq <= 0:
        return np.nan, np.nan
    t = (r12 - r13) * np.sqrt((n-1) * (1+r23)) / np.sqrt(denom_sq)
    p = 2 * stats.t.sf(abs(t), df=n-3)
    return t, p

n = len(df_sent)
t_st, p_st = steigers_test(r_sentiment, r_valence, r_sv, n)

print('=== H2: Lyrics sentiment vs. valence — hangisi daha iyi öngörüyor? ===')
print(f'Spearman r (sentiment vs popularity): {r_sentiment:.4f}  (p={p_rs:.4f})')
print(f'Spearman r (valence   vs popularity): {r_valence:.4f}  (p={p_rv:.4f})')
print(f'Sentiment–valence korelasyonu:        {r_sv:.4f}')
print(f"Steiger's test: t = {t_st:.3f},  p = {p_st:.4f}")
if p_st < 0.05:
    stronger = 'lyrics sentiment' if abs(r_sentiment) > abs(r_valence) else 'valence'
    print(f'Sonuç: H0 REDDEDİLDİ ✓ — iki korelasyon anlamlı şekilde farklı. Daha güçlü değişken: {stronger}')
else:
    print('Sonuç: H0 REDDEDİLEMEDİ — iki korelasyon arasında anlamlı fark yok.')

### H3: Ses ile sözlerin duygusu uyuşmayan şarkılar daha mı popüler?

In [ ]:
high_gap = df_sent[df_sent['gap_group'] == 'High gap']['popularity']
low_gap  = df_sent[df_sent['gap_group'] == 'Low gap']['popularity']

stat_h3, p_h3 = stats.mannwhitneyu(high_gap, low_gap, alternative='greater')

print('=== H3: Yüksek duygusal uyumsuzluk → daha yüksek popularity? ===')
print(f'High gap şarkılar: n={len(high_gap)}, medyan popularity = {high_gap.median():.1f}')
print(f'Low gap şarkılar:  n={len(low_gap)}, medyan popularity = {low_gap.median():.1f}')
print(f'Mann-Whitney U = {stat_h3:.0f},  p = {p_h3:.4f}')
if p_h3 < 0.05:
    print('Sonuç: H0 REDDEDİLDİ ✓ — uyumsuz şarkılar anlamlı şekilde daha popüler.')
else:
    print("Sonuç: H0 REDDEDİLEMEDİ — uyumsuzluk popularity'i anlamlı şekilde etkilemiyor.")

### Özet Tablo

In [ ]:
print('=' * 60)
print('HİPOTEZ TESTİ SONUÇLARI')
print('=' * 60)
results = [
    ('H1', 'Pozitif lyrics → daha popüler (Mann-Whitney U)', p_h1),
    ('H2', 'Sentiment vs. valence farkı (Steiger)',          p_st),
    ('H3', 'Yüksek gap → daha popüler (Mann-Whitney U)',     p_h3),
]
for name, desc, pval in results:
    karar = 'H0 REDDEDİLDİ ✓' if pval < 0.05 else 'H0 reddedilemedi'
    print(f'{name}: {desc}')
    print(f'     p = {pval:.4f}  →  {karar}')
    print()

---
## 7. En Fazla Uyumsuz Şarkılar (Bonus)

Sesi neşeli ama sözleri karanlık (veya tam tersi) olan şarkıların listesi:

In [ ]:
top_mismatch = df_sent.sort_values('valence_sentiment_gap', ascending=False)
show_cols = ['track_name', 'artist', 'popularity', 'valence', 'compound_sentiment', 'valence_sentiment_gap']
top_mismatch[show_cols].head(10).reset_index(drop=True)